In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
)
from sklearn.metrics import (
    classification_report, ConfusionMatrixDisplay, precision_recall_curve
)
from xgboost import XGBClassifier
import shap

In [ ]:
df = pd.read_csv('data/global_supply_chain_risk_2026.csv')
print(df.shape)
df.head()

In [ ]:
print('Missing values:\n', df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())
print('\nClass imbalance:')
print(df['Disruption_Occurred'].value_counts(normalize=True).rename('proportion'))

In [ ]:
plt.figure(figsize=(12, 7), dpi=100)
sns.heatmap(df.select_dtypes(include='number').corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Da
df['Date'] = pd.to_datetime(df['Date'])
df['year']        = df['Date'].dt.year
df['month']       = df['Date'].dt.month
df['day_of_week'] = df['Date'].dt.dayofweek
df = df.drop('Date', axis=1)

# ✅ FIX: delete ID before encoding
df = df.drop('Shipment_ID', axis=1)

# One-hot encoding 
cat_cols = df.select_dtypes(include='object').columns.tolist()
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# bool -> int
bool_cols = df.select_dtypes(include='bool').columns.tolist()
df[bool_cols] = df[bool_cols].astype(int)

df.info()

In [ ]:
TARGET = 'Disruption_Occurred'
X = df.drop(TARGET, axis=1)
y = df[TARGET]

# stratify=y on each step
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=42
)

print(f'Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}')

In [ ]:
selector = Pipeline([
    ('variance', VarianceThreshold(threshold=0.01)),
    ('kbest',    SelectKBest(f_classif, k=6)),
])
selector.fit(X_train, y_train)

X_train_f = pd.DataFrame(selector.transform(X_train),
                          columns=X_train.columns[selector['variance'].get_support()][selector['kbest'].get_support()])
X_val_f   = pd.DataFrame(selector.transform(X_val),   columns=X_train_f.columns)
X_test_f  = pd.DataFrame(selector.transform(X_test),  columns=X_train_f.columns)

# cv for feature selection
X_cv = pd.DataFrame(selector.transform(X_temp), columns=X_train_f.columns)

good_features = X_train_f.columns.tolist()
print('selected features:', good_features)

In [ ]:
neg, pos = np.bincount(y)
SCALE_POS = neg / pos
SCORING   = 'f1_macro'
CV        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
N_TRIALS  = 100

print(f'disbalance → 0: {neg}, 1: {pos}  |  scale_pos_weight = {SCALE_POS:.3f}')
print(f'Metric: {SCORING}')

In [ ]:
def run_study(name: str, objective, n_trials: int) -> optuna.Study:
    study = optuna.create_study(
        study_name=name,
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study

In [ ]:
def objective_lr(trial: optuna.Trial) -> float:
    solver = trial.suggest_categorical('solver', ['lbfgs', 'saga', 'liblinear'])

    if solver == 'lbfgs':
        penalty = trial.suggest_categorical('penalty_lbfgs', ['l2', None])
    elif solver == 'saga':
        penalty = trial.suggest_categorical('penalty_saga', ['l1', 'l2', 'elasticnet', None])
    else:
        penalty = trial.suggest_categorical('penalty_liblinear', ['l1', 'l2'])

    params = dict(
        solver=solver, penalty=penalty,
        C=trial.suggest_float('C', 1e-3, 1e2, log=True),
        max_iter=trial.suggest_int('max_iter', 200, 2000, step=200),
        class_weight='balanced',
        random_state=42,
    )
    if penalty == 'elasticnet':
        params['l1_ratio'] = trial.suggest_float('l1_ratio', 0.0, 1.0)

    pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(**params))])
    scores = cross_val_score(pipe, X_cv, y_temp, cv=CV, scoring=SCORING, n_jobs=-1)
    return scores.mean()

study_lr = run_study('logistic_regression', objective_lr, N_TRIALS)

In [ ]:
def objective_xgb(trial: optuna.Trial) -> float:
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 1000, step=50),
        max_depth=trial.suggest_int('max_depth', 2, 10),
        learning_rate=trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
        colsample_bylevel=trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        min_child_weight=trial.suggest_int('min_child_weight', 1, 10),
        gamma=trial.suggest_float('gamma', 0.0, 5.0),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        scale_pos_weight=trial.suggest_float(
            'scale_pos_weight', SCALE_POS * 0.5, SCALE_POS * 2.0
        ),
        eval_metric='logloss',
        tree_method='hist',
        random_state=42,
        n_jobs=-1,
    )
    model = XGBClassifier(**params)
    scores = cross_val_score(model, X_cv, y_temp, cv=CV, scoring=SCORING, n_jobs=-1)
    return scores.mean()


study_xgb = run_study('xgboost', objective_xgb, N_TRIALS)

In [ ]:
def build_lr(study: optuna.Study) -> Pipeline:
    p = study.best_params.copy()
    solver = p.pop('solver')
    penalty = None
    for key in list(p.keys()):
        if key.startswith('penalty_'):
            penalty = p.pop(key)
    p.update({'solver': solver, 'penalty': penalty,
              'class_weight': 'balanced', 'random_state': 42})
    return Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(**p))])


def build_xgb(study: optuna.Study) -> XGBClassifier:
    p = study.best_params.copy()
    p.update({'eval_metric': 'logloss', 'tree_method': 'hist',
              'random_state': 42, 'n_jobs': -1})
    return XGBClassifier(**p)

In [ ]:
final_lr  = build_lr(study_lr)
final_xgb = build_xgb(study_xgb)

final_lr.fit(X_train_f, y_train)
final_xgb.fit(X_train_f, y_train)
print(f'✓ LR trained.   Best {SCORING}: {study_lr.best_value:.5f}')
print(f'✓ XGB trained.   Best {SCORING}: {study_xgb.best_value:.5f}')

In [ ]:
def find_best_threshold_oof(model, X_oof, y_oof) -> float:
    """Selection of the best threshold based on OOF predictions."""
    probs = cross_val_predict(model, X_oof, y_oof, cv=CV, method='predict_proba')[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_oof, probs)
    f1_scores = np.where(
        (precisions + recalls) == 0, 0,
        2 * precisions * recalls / (precisions + recalls)
    )
    best_idx = np.argmax(f1_scores[:-1])
    thr = float(thresholds[best_idx])
    print(f'  OOF best threshold: {thr:.3f}  →  F1: {f1_scores[best_idx]:.4f}')
    return thr


print('LogisticRegression OOF threshold:')
thr_lr  = find_best_threshold_oof(final_lr,  X_train_f, y_train)
print('XGBClassifier OOF threshold:')
thr_xgb = find_best_threshold_oof(final_xgb, X_train_f, y_train)

In [ ]:
def evaluate(model, X_t, y_t, threshold: float, name: str) -> None:
    probs  = model.predict_proba(X_t)[:, 1]
    y_pred = (probs >= threshold).astype(int)
    print(f'\n{"-" * 55}')
    print(f'  {name}  (threshold = {threshold:.3f})')
    print(f'{"-" * 55}')
    print(classification_report(y_t, y_pred, digits=3))


evaluate(final_lr,  X_test_f, y_test, thr_lr,  'LogisticRegression — test')
evaluate(final_xgb, X_test_f, y_test, thr_xgb, 'XGBClassifier — test')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

probs_lr  = (final_lr.predict_proba(X_test_f)[:, 1] >= thr_lr).astype(int)
probs_xgb = (final_xgb.predict_proba(X_test_f)[:, 1] >= thr_xgb).astype(int)

ConfusionMatrixDisplay.from_predictions(y_test, probs_lr,  ax=axes[0], colorbar=False)
ConfusionMatrixDisplay.from_predictions(y_test, probs_xgb, ax=axes[1], colorbar=False)
axes[0].set_title('LogisticRegression')
axes[1].set_title('XGBClassifier')
plt.tight_layout()
plt.show()

In [ ]:
X_train_scaled = final_lr.named_steps['scaler'].transform(X_train_f)
X_test_scaled  = final_lr.named_steps['scaler'].transform(X_test_f)

lr_model   = final_lr.named_steps['clf']
explainer_lr = shap.LinearExplainer(lr_model, X_train_scaled)
shap_vals_lr = explainer_lr.shap_values(X_test_scaled)

shap.summary_plot(shap_vals_lr, X_test_scaled, feature_names=good_features)

In [ ]:
explainer_xgb = shap.TreeExplainer(final_xgb)
shap_vals_xgb = explainer_xgb.shap_values(X_test_f)

shap.summary_plot(shap_vals_xgb, X_test_f, feature_names=good_features)

In [ ]:
print('=' * 55)
print('  Best parameters — LogisticRegression')
print('=' * 55)
for k, v in study_lr.best_params.items():
    print(f'  {k}: {v}')

print('\n' + '=' * 55)
print('  Best parameters — XGBClassifier')
print('=' * 55)
for k, v in study_xgb.best_params.items():
    print(f'  {k}: {v}')